Version 0: Circuito que construye la superposición: Sum_{2^I}(|Index_i>|Window_i>|Grid(Irrelevant)>):
- Red de 1 dimensión.

In [ ]:
import time
import qiskit
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.visualization import plot_histogram
print(qiskit.__version__)

In [ ]:
N = 9
M = 2
W = N - M + 1
K = np.ceil(np.log2(W)).astype(int)
AUX = 1
print(f"Posibles soluciones W: {W}, qubits necesarios para representar W posiciones -> k: {K}")

In [ ]:
# Funcion que pone en superposicion los indices + la ventana
def superposition():
    for i in range(W):
        bits = [(i >> b) & 1 for b in range(K)]  # idx[0] es LSB, idx[k-1] es MSB -> little-endian

        # Activar control de igualdad idx == i
        for b in range(K):
            if bits[b] == 0:
                qc.x(idx[b])

        # XOR de la ventana seleccionada sobre m
        for j in range(M):
            controls = [idx[b] for b in range(K)] + [n[i + j]]
            qc.mcx(controls, m[j])

        # Deshacer X
        for b in range(K):
            if bits[b] == 0:
                qc.x(idx[b])

In [ ]:
def print_state_grouped(sv, title="SUPERPOSED STATES", tol=1e-10):
    print(f"\n============ {title} ============")

    total_qubits = int(np.log2(len(sv.data)))
    shown_qubits = total_qubits - N

    if shown_qubits < AUX + M + K:
        raise ValueError("Los tamaños de registros no cuadran con el número total de qubits mostrado.")

    filas = []

    for i, amp in enumerate(sv.data):
        if abs(amp) <= tol:
            continue

        bits_full = format(i, f"0{total_qubits}b")
        bits = bits_full[:shown_qubits]

        pos = 0
        aux_bits = bits[pos:pos + AUX]
        pos += AUX

        m_desc = bits[pos:pos + M]
        pos += M

        idx_bits = bits[pos:pos + K]   # binario normal para ordenar
        pos += K

        window_bits = m_desc[::-1]     # mostrar como m_0...m_{M-1}

        filas.append((idx_bits, aux_bits, window_bits, amp))

    filas.sort(key=lambda x: (x[0], x[2], x[3]))

    for idx_bits, aux_bits, window_bits, amp in filas:
        print(f"{amp.real: .4g} aux=|{aux_bits}> window=|{window_bits}> index=|{idx_bits}>")

In [ ]:
N = 9
M = 2
W = N - M + 1
K = np.ceil(np.log2(W)).astype(int)
AUX = 1
print(f"Posibles soluciones W: {W}, qubits necesarios para representar W posiciones -> k: {K}")

# n: posiciones libres/ocupadas
n = QuantumRegister(N, "n")
# m: trabajo (tamano M)
m = QuantumRegister(M, "m")
# idx: seleccion de ventana (tamano k)
idx = QuantumRegister(K, "i")
# aux: qubits auxiliares para operaciones de control
aux = QuantumRegister(1, "aux")

qc = QuantumCircuit(n, idx, m, aux)

# ---------------------------
# Estado inicial de ejemplo para n
qc.x(n[0])
qc.x(n[1])
qc.x(n[2])
qc.x(n[5])
qc.x(n[6])
qc.x(n[7])
qc.x(n[8])
# qc.x(n[13])
# ---------------------------

# Superposicion uniforme de indices
amps_idx = np.zeros(2**K, dtype=complex)
amps_idx[:W] = 1 / np.sqrt(W)
qc.initialize(amps_idx, idx)

for i in range(W):
        bits = [(i >> b) & 1 for b in range(K)]  # idx[0] es LSB, idx[k-1] es MSB -> little-endian

        # Activar control de igualdad idx == i
        for b in range(K):
            if bits[b] == 0:
                qc.x(idx[b])

        # XOR de la ventana seleccionada sobre m
        for j in range(M):
            controls = [idx[b] for b in range(K)] + [n[i + j]]
            qc.mcx(controls, m[j])

        # Deshacer X
        for b in range(K):
            if bits[b] == 0:
                qc.x(idx[b])
qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
sv.draw(output='latex', max_size=2**qc.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
print_state_grouped(sv)

In [ ]:
qc.x(aux)
qc.h(aux)
qc.barrier()
qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
sv.draw(output='latex', max_size=2**qc.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
sv = Statevector(qc)
print_state_grouped(sv)

In [ ]:
qc.cx(idx[0], aux)
qc.cx(idx[1], aux)
qc.barrier()
qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
print_state_grouped(sv)

In [ ]:
superposition()
qc.barrier()
qc.draw(output='mpl')

In [ ]:
qc.h(idx)
qc.barrier()
qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
print_state_grouped(sv)

In [ ]:
qc.h(aux)
qc.x(aux)
qc.barrier()
qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
print_state_grouped(sv)

In [ ]:
sv = Statevector(qc)
sv.draw(output='latex', max_size=2**qc.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum_platform",token="<IBM_QUANTUM_TOKEN>",instance="<IBM_QUANTUM_INSTANCE>",overwrite=True)
# service = QiskitRuntimeService()
# backend = service.least_busy(simulator=False, operational=True, min_num_qubits=16)
# # backend = FakeManilaV2()
# print(backend)

In [ ]:
# Transpilation
# qc.decompose().draw('mpl',fold=-1,idle_wires=False)

circuito lógico

   ↓

transpile

   ↓

circuito transpiled

   ↓

lowering / scheduling / backend instructions

   ↓

ISA circuit

In [ ]:
# # Convert to an ISA circuit.
# pm = generate_preset_pass_manager(backend=backend, optimization_level=2)
# isa_circuit = pm.run(qc)
# isa_circuit.draw("mpl", idle_wires=False)

In [ ]:
# pm_sampler = generate_preset_pass_manager(backend=backend, optimization_level=1) 
# isa_circuit_sampler = pm_sampler.run(qc) 
# isa_circuit_sampler.draw("mpl", idle_wires=False)

# # Create a sampler using the selected backend
# sampler = Sampler(mode=backend) 
# # Run the sampler primitive on ISA circuit for specified number of shots (1024)
# job_sampler = sampler.run([isa_circuit_sampler], shots=1024) 
# # Save the result of the job 
# result_sampler = job_sampler.result()

# counts = result_sampler[0].data.c.get_counts()

# # Plot the result
# plot_histogram(counts)